# 08 — Freezing the model for the prototype

Produces the artifact the prototype loads. Nothing here is exploratory — every choice was made and evidenced in an earlier notebook, and this one records them.

**What gets frozen**

| file | why the prototype needs it |
|---|---|
| `model.json` | the estimator, in XGBoost's version-portable format |
| `preprocessor.json` | the fitted feature pipeline, including the **train-only** amount cutoff and severity grid |
| `thresholds.json` | frozen decision threshold, severity ratio, band cutoffs and actions |
| `model_card.json` | what it was trained on, how it scores, and where it should not be trusted |

**Two decisions worth stating.**

The model is refit on **train + validation** so the shipped artifact uses all available data, but the decision threshold is the one selected on validation using the *train-only* model. Re-deriving a threshold on data the final model has seen would be selecting on the training set.

The final fit is **unweighted**. Notebook 04 tested weighting, severity-weighting and undersampling across four learners and twelve comparisons: nine were significantly worse than unweighted, three indistinguishable, none better. Shipping a cost-sensitive *training* method would contradict the study's own finding.

In [1]:
import json, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from momo_fraud import constants as C
from momo_fraud import data as D
from momo_fraud import evaluate as E
from momo_fraud import experiment as X
from momo_fraud import models as M
from momo_fraud import risk as RK
from momo_fraud import splits as S
from momo_fraud.features import FeatureBuilder, columns_for
from momo_fraud.predict import FraudScorer, build_thresholds

RESULTS = PROJECT_ROOT / "results"
ARTIFACTS = PROJECT_ROOT / "artifacts"

# Resolves to the validation-selected decision when present, falling back to
# the original test-selected one. Both name the same rung; the preference
# order exists so a future re-derivation cannot silently fall back.
decision = X.load_experimental_rung()
RUNG = decision["experimental_rung"]
LEARNER = "xgboost"
SEED, R = C.RANDOM_SEED, C.R_DEFAULT

print(f"rung {RUNG} | learner {LEARNER} | R {R:.2f} | lambda {RK.bayes_threshold(R):.6f}")

rung no_origin_balance | learner xgboost | R 773.70 | lambda 0.001291


In [2]:
df = D.load()
y = df[C.TARGET].to_numpy()
split = S.stratified_split(y)

builder = FeatureBuilder().fit(df.iloc[split.train])
X_all = builder.transform(df)
cols = columns_for(RUNG, list(X_all.columns))

# Threshold selection uses the train-only model on validation.
interim = M.make_learner(LEARNER, seed=SEED)
interim.fit(X_all.iloc[split.train][cols], y[split.train])
val_scores = interim.predict_proba(X_all.iloc[split.val][cols])[:, 1]

chosen = RK.optimal_threshold(y[split.val], val_scores, R)
print(f"threshold selected on validation: {chosen.threshold:.6f}")
print(f"  vs the closed-form Bayes threshold: {RK.bayes_threshold(R):.6f}")
print(f"  validation NER {chosen.ner:.4f}, recall {chosen.recall:.4f}")

threshold selected on validation: 0.001916
  vs the closed-form Bayes threshold: 0.001291
  validation NER 0.1147, recall 0.9042


In [3]:
# Held-out test performance, using that frozen threshold. This is what the
# model card reports -- the test set is touched once, here, and never tuned on.
test_scores = interim.predict_proba(X_all.iloc[split.test][cols])[:, 1]
held_out = E.full_report(y[split.test], test_scores,
                         threshold=chosen.threshold, r=R,
                         learner=LEARNER, rung=RUNG)

# Two different quantities, so name them separately. A raw top-k on the test
# split is not the same as an alerts-per-day capacity: the split holds ~15% of
# traffic over ~31 days, so 100/day corresponds to ~4,650 alerts here. Reporting
# one under the other's name overstates capacity by roughly 46x.
budgets_top_k = RK.budget_curve(y[split.test], test_scores)

test_fraction = len(split.test) / C.N_ROWS
n_days = C.MAX_STEP / C.STEPS_PER_DAY
budgets_per_day = {
    per_day: RK.recall_at_budget(
        y[split.test], test_scores,
        max(int(round(per_day * n_days * test_fraction)), 1))
    for per_day in C.REVIEW_BUDGETS
}
for key in ("pr_auc", "precision", "recall", "f1", "mcc", "ner", "n_flagged"):
    print(f"  {key:12s} {held_out[key]:.4f}" if isinstance(held_out[key], float)
          else f"  {key:12s} {held_out[key]:,}")

  pr_auc       0.8173
  precision    0.0582
  recall       0.8872
  f1           0.1093
  mcc          0.2246
  ner          0.1314
  n_flagged    18,776


In [4]:
# Final fit on train + validation.
final_rows = np.concatenate([split.train, split.val])
start = time.perf_counter()

final_model = M.make_learner(LEARNER, seed=SEED)
final_model.fit(X_all.iloc[final_rows][cols], y[final_rows])
print(f"final fit on {len(final_rows):,} rows in {time.perf_counter() - start:.0f}s")

thresholds = build_thresholds(decision_threshold=chosen.threshold, r=R,
                              feature_names=cols, rung=RUNG)
scorer = FraudScorer(model=final_model, builder=builder,
                     thresholds=thresholds, feature_names=cols)

for band, cutoff in thresholds["band_cutoffs"].items():
    print(f"  {band:9s} >= {cutoff:8.4f} -> {C.BAND_ACTIONS[band]}")

final fit on 5,408,226 rows in 165s
  Medium    >=   0.0129 -> monitor
  High      >=   0.1291 -> step_up_auth
  Critical  >=   1.2908 -> block


## The model card

Provenance, performance and — most importantly — the limitations. Anyone loading this artifact needs to know it was trained on synthetic data whose fraud is largely rule-recoverable, and what that does and does not license.

In [5]:
provenance = json.loads((RESULTS / "00_dataset_provenance.json").read_text())

model_card = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "task": "mobile money transaction fraud detection",
    "learner": LEARNER,
    "cost_sensitivity": "decision-level only (threshold); no class weighting or resampling",
    "data": {
        "dataset": provenance["dataset"],
        "csv_sha256": provenance["csv_sha256"],
        "n_transactions": provenance["n_transactions"],
        "prevalence": provenance["prevalence"],
        "trained_on_rows": int(len(final_rows)),
        "feature_set": RUNG,
        "features_excluded": decision["features_dropped"],
    },
    "decision": {
        "severity_ratio_R": float(R),
        "interpretation": "missing one fraud is treated as bad as R false alarms",
        "threshold": float(chosen.threshold),
        "selected_on": "validation split, using a train-only model",
    },
    "held_out_performance": {
        k: float(held_out[k]) for k in ("pr_auc", "roc_auc", "precision", "recall",
                                        "f1", "mcc", "brier", "ner", "alert_rate")
    },
    "recall_at_top_k_on_test_split": {
        str(k): v["recall"] for k, v in budgets_top_k.items()
    },
    "recall_at_alerts_per_day": {
        str(k): float(v) for k, v in budgets_per_day.items()
    },
    "budget_note": (
        f"The test split holds {test_fraction:.1%} of traffic over {n_days:.1f} "
        "days. 'alerts_per_day' figures are scaled accordingly; 'top_k' figures "
        "are raw counts on the split. Do not read one as the other."
    ),
    "limitations": [
        "Trained on PaySim, a synthetic simulator, not real mobile money traffic.",
        f"PaySim's fraud is largely recoverable by a three-clause boolean rule "
        f"(precision {decision['rule_precision']:.4f}, recall {decision['rule_recall']:.4f}), "
        "so results on the full feature set measure the simulator rather than fraud.",
        f"This model deliberately excludes origin-balance features for that reason; "
        f"it is trained on the '{RUNG}' feature set.",
        "ROC-AUC is reported for comparability only. At 0.129% prevalence it is "
        "near-insensitive to false positives and should not be used to rank models.",
        "Thresholds assume calibrated probabilities. Recalibrate before deploying "
        "against traffic whose prevalence differs from 0.129%.",
        "No fairness or demographic evaluation was performed; PaySim carries no "
        "demographic attributes.",
    ],
}

scorer.save(ARTIFACTS, model_card=model_card)
print(f"written to {ARTIFACTS}")
for path in sorted(ARTIFACTS.iterdir()):
    print(f"  {path.name:22s} {path.stat().st_size / 1024:>8.1f} KB")

written to C:\Users\siphe\source\repos\CodeHermez\momo-fraud-supervised-ml\artifacts
  model.json               1365.9 KB
  model_card.json             2.6 KB
  preprocessor.json          18.4 KB
  thresholds.json             0.7 KB


## Export fidelity

The check that matters: a bundle reloaded from disk must reproduce the in-memory model's scores **exactly**. If it does not, the prototype and this study disagree about what a transaction is worth.

In [6]:
reloaded = FraudScorer.load(ARTIFACTS, with_explainer=False)
sample = df.iloc[split.test[:5000]]

before = scorer.score_batch(sample)["probability"].to_numpy()
after = reloaded.score_batch(sample)["probability"].to_numpy()
np.testing.assert_allclose(before, after, rtol=1e-6)

max_diff = float(np.abs(before - after).max())
print(f"PASS - reloaded bundle reproduces {len(sample):,} scores "
      f"(max difference {max_diff:.2e})")

# And the single-transaction path must agree with the batch path.
for position in (0, 1234, 4999):
    single = reloaded.score(sample.iloc[position].to_dict(), explain=False)
    assert abs(single["probability"] - after[position]) < 1e-6
print("PASS - single-transaction path matches the batch pipeline")

PASS - reloaded bundle reproduces 5,000 scores (max difference 0.00e+00)
PASS - single-transaction path matches the batch pipeline


## What the prototype will see

The contract, demonstrated on real transactions — one fraudulent, one legitimate.

In [7]:
demo = FraudScorer.load(ARTIFACTS, with_explainer=True)
test_frame = df.iloc[split.test]

examples = {
    "a fraudulent transfer": test_frame[test_frame[C.TARGET] == 1].iloc[0],
    "a legitimate payment": test_frame[test_frame[C.TARGET] == 0].iloc[0],
}

for label, row in examples.items():
    result = demo.score(row.to_dict())
    print(f"\n--- {label} (actual isFraud = {int(row[C.TARGET])}) ---")
    print(f"  risk_score {result['risk_score']:.3f} / 100")
    print(f"  band       {result['band']}  ->  {result['action']}")
    print(f"  flagged    {result['flagged']}  (threshold {result['decision_threshold']:.5f})")
    for factor in result.get("top_factors", [])[:3]:
        print(f"    {factor['feature']:18s} {factor['contribution']:+.4f}  "
              f"{factor['direction']}")

C:\Users\siphe\source\repos\CodeHermez\momo-fraud-supervised-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- a fraudulent transfer (actual isFraud = 1) ---
  risk_score 100.000 / 100
  band       Critical  ->  block
  flagged    True  (threshold 0.00192)
    oldbalanceDest     -24.4003  decreases risk
    destZeroAfter      +17.4702  increases risk
    newbalanceDest     +14.7410  increases risk

--- a legitimate payment (actual isFraud = 0) ---
  risk_score 0.000 / 100
  band       Low  ->  allow
  flagged    False  (threshold 0.00192)
    type_TRANSFER      -8.8868  decreases risk
    type_PAYMENT       -4.5041  decreases risk
    destZeroAfter      +3.3035  increases risk


In [8]:
print("Load it from the prototype with:\n")
print("    from momo_fraud.predict import FraudScorer")
print("    scorer = FraudScorer.load('artifacts')")
print("    result = scorer.score({")
print("        'step': 1, 'type': 'TRANSFER', 'amount': 181.0,")
print("        'oldbalanceOrg': 181.0, 'newbalanceOrig': 0.0,")
print("        'oldbalanceDest': 0.0, 'newbalanceDest': 0.0,")
print("    })\n")
print("Returns risk_score (0-100), band, recommended action, flagged, and")
print("top_factors explaining the score. score_batch() takes a DataFrame.")

Load it from the prototype with:

    from momo_fraud.predict import FraudScorer
    scorer = FraudScorer.load('artifacts')
    result = scorer.score({
        'step': 1, 'type': 'TRANSFER', 'amount': 181.0,
        'oldbalanceOrg': 181.0, 'newbalanceOrig': 0.0,
        'oldbalanceDest': 0.0, 'newbalanceDest': 0.0,
    })

Returns risk_score (0-100), band, recommended action, flagged, and
top_factors explaining the score. score_batch() takes a DataFrame.
